# 🚀 Pipeline: Guardado, Exportación y Despliegue

**Propósito:** Aprender a guardar modelos entrenados, exportarlos a formatos portables
(TFLite, ONNX) y prepararlos para producción (FastAPI, HuggingFace Spaces).

---
## 🎯 ¿Por qué es importante el despliegue?

Un modelo entrenado que no se despliega no genera valor. El despliegue permite:
- **Inferencia en tiempo real** (APIs REST)
- **Ejecución en dispositivos móviles** (TensorFlow Lite)
- **Interoperabilidad** (ONNX → PyTorch, MXNet, etc.)
- **Escalabilidad** (múltiples peticiones concurrentes)

| Formato | Uso | Peso | Velocidad |
|---------|-----|------|-----------|
| `.keras` | Guardar/cargar en Keras | Original | — |
| `.tflite` | Móvil, edge, web | Reducido | Optimizado |
| `.onnx` | Intercambio entre frameworks | Variable | Multi-backend |

In [ ]:
# MACHOTE — Setup universal
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..')))
from modulo4_libreria import *

INFO = setup_completo()

In [ ]:
# MACHOTE — Entrenar modelo rápido en MNIST

print("Cargando MNIST...")
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

X_train = X_train.astype('float32').reshape(-1, 28, 28, 1) / 255.0
X_test  = X_test.astype('float32').reshape(-1, 28, 28, 1) / 255.0

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42
)

modelo = crear_cnn((28,28,1), 10, filtros=[32, 64])
hist = compilar_y_entrenar(
    modelo, X_train, y_train, X_val, y_val,
    num_clases=10, epochs=5, batch_size=64, verbose=1
)

# Guardar predicciones de referencia
preds_original = modelo.predict(X_test[:10], verbose=0)
print("\n✅ Modelo rápido entrenado en MNIST")

In [ ]:
# MACHOTE — Guardar y cargar con formato .keras

ruta_guardado = INFO['rutas']['modelos']

# Guardar
ruta = guardar_modelo(modelo, 'mnist_cnn', ruta_modelos=ruta_guardado)

# Cargar
modelo_cargado = cargar_modelo('mnist_cnn', ruta_modelos=ruta_guardado)

# Verificar que las predicciones coinciden
if modelo_cargado is not None:
    preds_cargado = modelo_cargado.predict(X_test[:10], verbose=0)
    coinciden = np.allclose(preds_original, preds_cargado, atol=1e-6)
    print(f"\n🔍 ¿Predicciones idénticas? {'✅ Sí' if coinciden else '❌ No'}")
    if not coinciden:
        diff = np.abs(preds_original - preds_cargado).max()
        print(f"   Diferencia máxima: {diff:.2e}")

---
## 📱 Exportación a TensorFlow Lite

TFLite está optimizado para dispositivos móviles, embebidos y Web (TensorFlow.js).
Reduce el tamaño del modelo y acelera la inferencia.

In [ ]:
# MACHOTE — Exportar a TFLite

TFLITE_DISPONIBLE = False
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(modelo)
    tflite_model = converter.convert()
    
    ruta_tflite = os.path.join(ruta_guardado, 'mnist_cnn.tflite')
    with open(ruta_tflite, 'wb') as f:
        f.write(tflite_model)
    
    TFLITE_DISPONIBLE = True
    
    peso_keras = os.path.getsize(ruta) / 1024
    peso_tflite = os.path.getsize(ruta_tflite) / 1024
    print(f"✅ Modelo exportado a TFLite: {ruta_tflite}")
    print(f"   Tamaño .keras: {peso_keras:.1f} KB")
    print(f"   Tamaño .tflite: {peso_tflite:.1f} KB")
    print(f"   Reducción: {(1 - peso_tflite/peso_keras)*100:.1f}%")
    
except Exception as e:
    print(f"⚠️ Error exportando a TFLite: {e}")
    print("   Posible causa: TensorFlow version incompatible con Windows.")

In [ ]:
# MACHOTE — Verificar inferencia con TFLite (si se exportó)

if TFLITE_DISPONIBLE:
    import numpy as np
    
    interpreter = tf.lite.Interpreter(model_path=ruta_tflite)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    # Inferencia con TFLite
    test_input = X_test[:5].astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], test_input)
    interpreter.invoke()
    preds_tflite = interpreter.get_tensor(output_details[0]['index'])
    
    # Comparar con modelo original
    preds_keras = modelo.predict(test_input, verbose=0)
    
    print("🔍 Comparación Keras vs TFLite (primeras 5 muestras):")
    print(f"   Keras:  {np.argmax(preds_keras, axis=1)}")
    print(f"   TFLite: {np.argmax(preds_tflite, axis=1)}")
    print(f"   Coinciden: {np.array_equal(np.argmax(preds_keras, axis=1), np.argmax(preds_tflite, axis=1))}")

---
## 🔄 Exportación a ONNX

ONNX (Open Neural Network Exchange) permite usar el modelo en PyTorch, MXNet,
Windows ML, etc. Requiere `tf2onnx`.

In [ ]:
# MACHOTE — Exportar a ONNX

ONNX_DISPONIBLE = False
try:
    import tf2onnx
    
    ruta_onnx = os.path.join(ruta_guardado, 'mnist_cnn.onnx')
    
    # Especificar shape de entrada
    spec = (tf.TensorSpec((None, 28, 28, 1), tf.float32, name='input'),)
    modelo_proto, _ = tf2onnx.convert.from_keras(modelo, input_signature=spec)
    
    with open(ruta_onnx, 'wb') as f:
        f.write(modelo_proto.SerializeToString())
    
    ONNX_DISPONIBLE = True
    print(f"✅ Modelo exportado a ONNX: {ruta_onnx}")
    print(f"   Tamaño: {os.path.getsize(ruta_onnx) / 1024:.1f} KB")
    
except ImportError:
    print("⚠️ tf2onnx no instalado. Para exportar a ONNX:")
    print("   pip install tf2onnx onnx")
except Exception as e:
    print(f"⚠️ Error exportando a ONNX: {e}")

---
## 🌐 FastAPI — API REST para servir el modelo (código conceptual)

El siguiente código es ilustrativo para crear un microservicio con FastAPI.
Se ejecutaría en un archivo aparte (`app.py`), no en el notebook.

```python
# app.py — FastAPI para servir modelo MNIST
from fastapi import FastAPI, UploadFile
import tensorflow as tf
import numpy as np
from PIL import Image
import io

app = FastAPI()
modelo = tf.keras.models.load_model('modelos/mnist_cnn.keras')

@app.post('/predict')
async def predict(file: UploadFile):
    img = Image.open(io.BytesIO(await file.read())).convert('L').resize((28, 28))
    arr = np.array(img).reshape(1, 28, 28, 1).astype('float32') / 255.0
    pred = modelo.predict(arr, verbose=0)
    clase = int(np.argmax(pred[0]))
    confianza = float(np.max(pred[0]))
    return {'digito': clase, 'confianza': confianza}
```

Para ejecutar:
```bash
pip install fastapi uvicorn python-multipart pillow
uvicorn app:app --reload --port 8000
```

Luego enviar petición:
```bash
curl -X POST -F "file=@digito.png" http://localhost:8000/predict
```

---
## 🤗 HuggingFace Spaces — Guía de despliegue

HuggingFace Spaces permite alojar demos interactivas de ML gratis (CPU o GPU).

### Pasos:

1. **Crear cuenta** en [huggingface.co](https://huggingface.co)
2. **Crear un Space**: New Space → nombre → SDK: Gradio / Streamlit
3. **Subir modelo**: Opción A — usar `guardar_modelo()` y subir el `.keras`
4. **Estructura del Space:**
   ```
   space/
   ├── app.py          # Código de la interfaz
   ├── requirements.txt # tensorflow, gradio, numpy, pillow
   ├── modelos/
   │   └── mnist_cnn.keras
   └── README.md
   ```
5. **Ejemplo app.py con Gradio:**
   ```python
   import gradio as gr
   import tensorflow as tf
   import numpy as np
   
   modelo = tf.keras.models.load_model('modelos/mnist_cnn.keras')
   
   def clasificar(img):
       img = img.reshape(1, 28, 28, 1).astype('float32') / 255.0
       pred = modelo.predict(img, verbose=0)[0]
       return {str(i): float(p) for i, p in enumerate(pred)}
   
   gr.Interface(
       fn=clasificar,
       inputs=gr.Sketchpad(shape=(28, 28), invert_colors=True),
       outputs=gr.Label(num_top_classes=3),
       title='Clasificador MNIST'
   ).launch()
   ```
6. **Subir**: git push al Space → automáticamente se despliega

---
## ✅ Checklist de despliegue

- [ ] Modelo entrenado y evaluado (accuracy > threshold)
- [ ] Modelo guardado en `.keras`
- [ ] Preprocesamiento idéntico al entrenamiento
- [ ] Exportado a TFLite (móvil) y/o ONNX (portabilidad)
- [ ] Pruebas de inferencia: original == cargado == exportado
- [ ] API documentada (FastAPI + Swagger)
- [ ] Manejo de errores (imagen inválida, timeout)
- [ ] Logging de peticiones y respuestas
- [ ] Escalabilidad: ¿cuántas peticiones por segundo?
- [ ] Monitoreo: latencia, errores, drift del modelo

---
*Pipeline creado para el Diplomado en Redes Neuronales — Módulo 4* 🧠

In [ ]:
# MACHOTE — Resumen de archivos exportados

print("📂 Archivos generados en esta sesión:")
print("=" * 50)

for f in os.listdir(ruta_guardado):
    ruta_f = os.path.join(ruta_guardado, f)
    tam = os.path.getsize(ruta_f) / 1024
    print(f"   📄 {f:<30s} {tam:>8.1f} KB")

print("=" * 50)
print(f"✅ Total: {len(os.listdir(ruta_guardado))} archivo(s)")

In [ ]:
# TODO: Tareas de despliegue
# 1. Sube el modelo exportado a HuggingFace Spaces
# 2. Crea un archivo app.py con Gradio (usa el código conceptual arriba)
# 3. Prueba la API de FastAPI localmente
# 4. ¿Qué pasa si envías una imagen que no es un dígito?
# Pregunta: ¿Cuánto tarda la inferencia en TFLite vs Keras?